# Track

Here we demonstrate the impacts of some of the {func}`tams.track` options.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np

import tams

import warnings

We load a pre-identified dataset so we can skip to the tracking stage.

In [ ]:
ce = tams.data.load_example("mpas-regridded-identify")
ce

In [ ]:
def geoax(*, proj=None):
    if proj is None:
        proj = ccrs.PlateCarree()
    _, ax = plt.subplots(figsize=(8, 5), subplot_kw=dict(projection=proj), layout="constrained")
    ax.gridlines(draw_labels=True)
    ax.add_feature(cfeature.LAND)
    return ax

ce.plot(ec="C0", fc="none", alpha=0.05, ax=geoax(), transform=ccrs.PlateCarree())

Note that these CE shapes haven't had convex hull applied (compare to below).

In [ ]:
ce.convex_hull.plot(ec="C0", fc="none", alpha=0.05, ax=geoax(), transform=ccrs.PlateCarree())

We split up the single dataframe into one dataframe per time.
(In the future TAMS may do this automatically.)

In [ ]:
times, ces = zip(*[(time, ce) for time, ce in ce.groupby("time")])
print(len(times))

In [ ]:
%%time

ce = tams.track(ces, times)

In [ ]:
ce.plot(fc="none", alpha=0.1, column="mcs_id", cmap="tab10", ax=geoax(), transform=ccrs.PlateCarree())

In [ ]:
# MCS average CEs per time (most often just one)
gb = ce.groupby("mcs_id")
(gb.size() / gb.time.nunique()).value_counts().sort_index(ascending=False)

In [ ]:
colors = plt.get_cmap("tab10").colors


def plot_tracks(ce, *, nt_threshold=10, ces=True, ax=None):
    if ax is None:
        ax = geoax()

    if ces:
        ce.plot(ec="C0", fc="none", alpha=0.025, ax=ax, transform=ccrs.PlateCarree())
        # TODO: match track colors? maybe just for the tracks meeting the threshold?

    n = 0
    for i, (mcs_id, g) in enumerate(ce.groupby("mcs_id")):
        c = colors[n % len(colors)]
        nt = g.time.nunique()
        if nt < nt_threshold:
            continue
        if nt <= 2:
            alpha = [1]
        else:
            alpha = np.linspace(0.1, 1.0, nt - 1)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            track = g.dissolve("time").centroid

        for j, (a, b) in enumerate(zip(track.iloc[:-1], track.iloc[1:])):
            ax.plot([a.x, b.x], [a.y, b.y], ls="-", c=c, alpha=alpha[j], transform=ccrs.PlateCarree())
        ax.annotate(mcs_id, (b.x, b.y), c=c, size=8, alpha=0.9, transform=ccrs.PlateCarree())

        n += 1

    n_tot = ce.mcs_id.nunique()
    ax.text(
        0.005, 0.985, f"{n}/{n_tot} tracks ($n_t > {nt_threshold}$) shown", 
        ha="left", va="top", transform=ax.transAxes,
    )
    # TODO: label avg dur, avg n_ce, avg distance per time step?


plot_tracks(ce, nt_threshold=6)

In [ ]:
plot_tracks(tams.track(ces, times, overlap_norm="min"), nt_threshold=6)

In [ ]:
plot_tracks(tams.track(ces, times, overlap_threshold=0.25), nt_threshold=6)

In [ ]:
plot_tracks(tams.track(ces, times, largest=True), nt_threshold=6)

In [ ]:
tams.plot_tracked(tams.track(ces, times, largest=True).query("mcs_id == 500"), label="none")